# 提示詞工程應用範例 - 社群留言情緒分析

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例  
此範例會需要短時間反覆大量調用LLM，使用免費的Gemini API會被限制流量而失敗。  
因此**建議選擇LM Studio中本地端開源模型。**

## Gemini

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-12b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

## 測試模型

In [4]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("機器學習的定義")
]
result = llm.invoke(messages)
print(result.content)

## 機器學習 (Machine Learning) 的定義

機器學習是**人工智能（AI）的一個分支，它使計算機系統能夠從數據中學習，而無需進行顯式編程。**  換句話說，我們不是直接告訴電腦如何執行某項任務，而是給它提供大量數據，讓它自己找出規律和模式，並根據這些規律做出預測或決策。

更詳細地解釋：

* **傳統的編程 (Traditional Programming):**
    * 我們明確地定義規則和指令，告訴電腦如何處理輸入並產生輸出。
    * 程式碼是固定的，如果需要改變行為，必須修改程式碼。
* **機器學習 (Machine Learning):**
    * 我們提供數據集（包含輸入和期望的輸出）。
    * 演算法分析數據，找出數據中的模式和關係。
    * 演算法建立一個模型，這個模型可以根據新的輸入預測輸出。
    * 模型會隨著更多數據的輸入而不斷改進。

**核心概念：**

* **數據 (Data):**  機器學習的燃料。可以是任何形式的信息，例如圖像、文本、數字等。
* **演算法 (Algorithm):** 用於分析數據並建立模型的數學公式或程序。
* **模型 (Model):** 根據數據訓練出來的表示數據規律的函數或結構。
* **訓練 (Training):** 使用數據來調整模型的參數，使其能夠做出準確的預測。
* **預測 (Prediction):**  使用訓練好的模型對新的、未見過的數據進行預測。

**簡單的比喻：**

想像一下你要教一個孩子分辨貓和狗。

* **傳統編程:** 你會告訴孩子：“貓有尖耳朵，狗有垂耳，貓的鼻子短，狗的鼻子長…”  如果孩子遇到一隻耳朵半立半垂的貓，就不知道該如何判斷了。
* **機器學習:** 你給孩子看大量的貓和狗的照片，讓孩子自己找出貓和狗之間的區別。 經過多次觀察後，孩子就能夠根據照片中的特徵來分辨貓和狗，即使遇到新的、不熟悉的貓或狗也能夠正確判斷。

**總結來說，機器學習的核心目標是：**  **從數據中學習，自動建立模型，並利用該模型進行預測或決策。**


希望這個解釋能幫助你理解機器學習的定義！



# 開始範例

## 範例測試用資料

In [ ]:
! pip install -U datasets

In [4]:
from datasets import load_dataset
# 載入 IMDB 資料集
dataset = load_dataset("stanfordnlp/imdb")
dataset

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [12]:
# 檢視單筆資料
dataset['test'][0] #這裡的數字可以填入資料的位置

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

## 測試提示詞與模型是否能正確預測

In [13]:
from langchain.messages import HumanMessage, SystemMessage
system_prompt = '''
請分析使用者的留言內容屬於正面情緒或是負面情緒？
輸出值為0或1。1代表正面,0代表負面
'''

user_prompt = dataset['test'][0]['text']
messages = [
    SystemMessage(system_prompt),
    HumanMessage(user_prompt)
]
result = llm.invoke(messages)
print(result.content)

0



## 計算IMDB留言情緒分析的正確率

In [14]:
from langchain.messages import HumanMessage, SystemMessage

def SentimentAnalysis(text) -> int:
    system_prompt = '''
    請分析使用者的留言內容屬於正面情緒或是負面情緒？
    直接輸出分數，分數為0或1。1代表正面,0代表負面
    除了分數，不要輸出其他的評論
    '''    
    user_prompt = text
    messages = [
        SystemMessage(system_prompt),
        HumanMessage(user_prompt)
    ]
    result = llm.invoke(messages)
    return int(result.content) #將模型輸出的文字轉成數值

SentimentAnalysis(dataset['test'][0]['text'])

0

In [44]:
from tqdm import tqdm
data_test = dataset['test'] # 使用測試組資料來測試
test_len = len(data_test) # 測試組資料的資料筆數
test_len = 1000 # 全部的測試組資料有25000筆，只測試前面1000筆

correct_count = 0
with tqdm(total=test_len, desc="社群情緒分析處理中") as pbar:
    for i in range(test_len):
        text = data_test[i]["text"]
        label = data_test[i]["label"]

        predict = SentimentAnalysis(text)
        if label == predict:
            correct_count = correct_count + 1
        
        # 動態更新進度條後方訊息
        pbar.set_postfix({"正確率": (correct_count/(i+1))})
        pbar.update(1)

社群情緒分析處理中: 100%|████████████████████████████████████████████| 1000/1000 [04:50<00:00,  3.45it/s, 正確率=0.941]/s]
